In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
#import utils
%run /Workspace/Users/utkarshadlakha1702@gmail.com/DataBricks_DataWarehousing/consolidated_pipeline/1_setup/Utilities


In [0]:
print(bronze_schema)

In [0]:
#creating widgets for better visiblity
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
print(catalog,data_source)
#displaying the

In [0]:
# read data path from s3
base_path = f's3://child-company-dbproj/{data_source}/*.csv'
df = (spark.read.format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(base_path)
      .withColumn("read_time", F.current_timestamp())
      .select("*","_metadata.file_name","_metadata.file_size")
      )
display(df.limit(10))

In [0]:
# Write all data as is to bronze layer
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

display(df.limit(10))